# 🧰 AI Technician Trainer

**Flow:** Enter Name & ID → Select Server Type → Read structured summary → Take 5 MCQ quiz → Score saved to Excel

> Run cells top-to-bottom once. Re-run Cell 4 only if PDFs change.

In [ ]:
# Cell 1 – Install dependencies
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters chromadb pypdf gradio openai openpyxl


In [ ]:
# Cell 2 – API Key
import os

# Option A: set directly (not recommended for sharing)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Option B: Colab secret (recommended)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    # Fallback: prompt input
    import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")
    print("✅ API key set.")


In [ ]:
from typing import Optional
# Cell 3 – Load PDFs and tag with server_type metadata
from pathlib import Path
import re
from langchain_community.document_loaders import PyPDFLoader

# ── Configure: point this at your PDF folder ──────────────────────────────
PDF_DIR = Path("/content/sample_data/training_material/")
# ──────────────────────────────────────────────────────────────────────────

def infer_server_type(filename: str) -> Optional[str]:
    """Extract A/B/C/D from filenames like 'Server_Type_A_...' ."""
    m = re.search(r"Type[\s_]*([ABCD])", filename, re.IGNORECASE)
    return m.group(1).upper() if m else None

docs = []
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(f"No PDFs found in {PDF_DIR}. Upload the 4 server PDFs first.")

for pdf_path in pdf_files:
    stype = infer_server_type(pdf_path.name)
    loaded = PyPDFLoader(str(pdf_path)).load()
    for d in loaded:
        d.metadata.update({
            "source_file": pdf_path.name,
            "server_type": stype,
            "server_label": f"Server Type {stype}" if stype else "Unknown",
            "document_type": "assembly_guide",
        })
    docs.extend(loaded)
    print(f"  Loaded {len(loaded):>3} pages  ←  {pdf_path.name}  (server_type={stype})")

print(f"\n✅ Total pages loaded: {len(docs)}")


In [ ]:
# Cell 4 – Chunk, embed, and persist ChromaDB
# ⚠️  Re-run this cell only when the source PDFs change.
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

CHROMA_DIR = "chroma_server_training"

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(docs)
print(f"Chunks created: {len(chunks)}")

emb = OpenAIEmbeddings(model="text-embedding-3-small")  # cheaper + accurate
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=emb,
    persist_directory=CHROMA_DIR,
)
vectorstore.persist()
print(f"✅ ChromaDB persisted to '{CHROMA_DIR}' with {vectorstore._collection.count()} vectors.")


In [ ]:
# Cell 5 – Load existing ChromaDB (fast path after first build)
# Run this instead of Cell 4 on subsequent sessions to skip re-embedding.
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

CHROMA_DIR = "chroma_server_training"
emb = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=emb,
)
print(f"✅ Loaded ChromaDB: {vectorstore._collection.count()} vectors.")


In [ ]:
from typing import Optional
# Cell 6 – Retriever + LLM setup
from langchain_openai import ChatOpenAI

def get_retriever(server_type: Optional[str] = None, k: int = 6):
    """
    Returns a Chroma retriever, optionally filtered to one server type.
    server_type: 'A' | 'B' | 'C' | 'D' | None (all)
    """
    search_kwargs = {"k": k}
    if server_type and server_type in ("A", "B", "C", "D"):
        search_kwargs["filter"] = {"server_type": server_type}
    return vectorstore.as_retriever(search_type="similarity", search_kwargs=search_kwargs)

# Shared LLM instance (temperature=0 for deterministic answers/grading)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ Retriever factory and LLM ready.")


In [ ]:
# Cell 7 – Shared helper utilities
import json
from typing import Any

def format_docs(docs) -> str:
    """Concatenate retrieved Document chunks into a single context string."""
    return "\n\n".join(
        f"[Source: {d.metadata.get('source_file','?')} | Page: {d.metadata.get('page','?')} | Server: {d.metadata.get('server_type','?')}]\n{d.page_content}"
        for d in docs
    )

def safe_json_loads(text: str) -> dict:
    """
    Best-effort JSON parser.
    Strips markdown fences and finds the outermost {...} block.
    """
    text = text.strip()
    # Strip ```json ... ``` fences
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"```\s*$", "", text, flags=re.MULTILINE)
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    # Fallback: extract first {...} block
    start = text.find("{")
    end   = text.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    raise ValueError(f"Could not parse JSON.\nRaw output:\n{text[:500]}")

import re  # needed for safe_json_loads
print("✅ Helper utilities ready.")


In [ ]:
from typing import Optional
# Cell 8 – RAG Q&A chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

def build_rag_chain(server_type: Optional[str] = None, k: int = 6):
    """
    Builds a RAG chain for open-ended Q&A.
    Answers in the same language as the question.
    """
    retriever = get_retriever(server_type=server_type, k=k)

    prompt = ChatPromptTemplate.from_template(
        "You are a careful AI assistant for manufacturing technicians.\n"
        "Use ONLY the provided context to answer.\n"
        "If the answer is not in the context, say you do not know.\n"
        "IMPORTANT: Answer in the SAME LANGUAGE as the user's question.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer (step-by-step, clear, safety-focused):"
    )

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
    )
    return chain

def answer_qa(question: str, server_type: Optional[str] = None) -> str:
    """High-level wrapper – returns plain string answer."""
    # Prepend explicit language instruction to the question so the model
    # always detects and mirrors the input language, even for voice input.
    augmented = (
        f"IMPORTANT: Detect the language of this question and answer in that EXACT same language.\n"
        f"Question: {question}"
    )
    chain = build_rag_chain(server_type=server_type, k=6)
    result = chain.invoke(augmented)
    return result.content if hasattr(result, "content") else str(result)

print("✅ RAG Q&A chain ready.")


In [ ]:
# Cell 9 – Training: Structured assembly summary
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def generate_training_summary(server_type: str, k: int = 20) -> str:
    """
    Generates a short, structured assembly summary from the server documentation.
    Uses strict rules to keep it well under the length of the original PDF.
    """
    retriever = get_retriever(server_type=server_type, k=k)
    retrieved_docs = retriever.invoke(
        f"Complete assembly procedure for Server Type {server_type}"
    )
    context = format_docs(retrieved_docs)

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are creating a SHORT technician quick-reference guide from a server assembly SOP.\n"
         "STRICT RULES:\n"
         "- Use ONLY information from the provided context. Do NOT invent anything.\n"
         "- Total output must be under 600 words.\n"
         "- Use clear markdown headings (##) and bullet points.\n"
         "- Do NOT repeat the same information twice.\n"
         "- Be concise — a busy technician must be able to read this in under 3 minutes.\n"
         "- Do NOT include warnings or disclaimers not present in the context."),
        ("human",
         "Context:\n{context}\n\n"
         "Write a short structured summary for Server Type {server_type} assembly using EXACTLY these sections:\n\n"
         "## Overview\n"
         "(2-3 sentences: what this server is and the purpose of assembly)\n\n"
         "## Tools & Parts Required\n"
         "(bullet list, max 8 items)\n\n"
         "## Assembly Steps\n"
         "(numbered list, max 12 steps, each step max 1 sentence)\n\n"
         "## Safety & Important Notes\n"
         "(bullet list, max 5 points)\n\n"
         "## Final Checks\n"
         "(bullet list, max 5 items to verify before sign-off)\n"
        )
    ])

    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "server_type": server_type})

print("✅ Training summary generator ready.")


In [ ]:
# Cell 14 – Quiz: MCQ builder
import random

QUIZ_FOCUS_PROMPTS = [
    "GPU installation, power cables, risers, PCIe seating",
    "ESD precautions, safety steps, common mistakes",
    "power supplies, cabling, airflow, fan modules",
    "storage installation (NVMe, SATA/SAS) and checks",
    "rack mounting, rail kit, power-on checks, BIOS/firmware",
    "cable management, connector types, torque specs",
    "final inspection, post-assembly verification steps",
]

def build_quiz(server_type: str, num_q: int = 5, k: int = 14) -> dict:
    """
    Builds a fresh MCQ quiz from the server's documentation.
    Returns: {"questions": [{question, options[4], correct_index, explanation}, ...]}
    """
    retriever = get_retriever(server_type=server_type, k=k)
    focus = random.choice(QUIZ_FOCUS_PROMPTS)
    retrieval_query = (
        f"Server Type {server_type} assembly – {focus}. "
        "Provide specific steps, part names, and safety requirements from the SOP."
    )
    retrieved_docs = retriever.invoke(retrieval_query)
    context = format_docs(retrieved_docs)

    # Use slightly elevated temperature for question variety
    try:
        llm_quiz = llm.bind(temperature=0.6)
    except Exception:
        llm_quiz = llm

    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are creating a certification quiz for a manufacturing technician.\n"
         "Rules:\n"
         "- Use ONLY the provided context.\n"
         "- Do NOT invent steps.\n"
         "- Each question must have EXACTLY 4 options.\n"
         "- EXACTLY ONE option is correct.\n"
         "- Keep questions practical and assembly-focused.\n"
         "- Return ONLY valid JSON."),
        ("human",
         "Context:\n{context}\n\n"
         "Create exactly {num_q} MCQ questions for Server Type {server_type}.\n"
         "Schema:\n"
         '{{"questions": [{{"question": "...", "options": ["...","...","...","..."], "correct_index": 0, "explanation": "..."}}]}}'
        )
    ])

    chain = prompt | llm_quiz | StrOutputParser()
    raw  = chain.invoke({"context": context, "num_q": num_q, "server_type": server_type})
    data = safe_json_loads(raw)

    qs = data.get("questions", [])
    if not isinstance(qs, list) or len(qs) < num_q:
        raise ValueError("Quiz generation failed — not enough questions returned. Try again.")

    cleaned = []
    for q in qs[:num_q]:
        opts = q.get("options", [])
        if not isinstance(opts, list) or len(opts) != 4:
            continue
        ci = q.get("correct_index", 0)
        if not isinstance(ci, int) or not (0 <= ci <= 3):
            ci = 0
        cleaned.append({
            "question":      str(q.get("question",    "")).strip(),
            "options":       [str(o).strip() for o in opts],
            "correct_index": ci,
            "explanation":   str(q.get("explanation", "")).strip(),
        })

    if len(cleaned) < num_q:
        raise ValueError(f"Quiz only produced {len(cleaned)}/{num_q} valid questions. Click 'Start Quiz' again.")

    return {"questions": cleaned[:num_q]}

print("✅ Quiz builder ready.")


In [ ]:
# Cell 15 – Voice: Whisper transcription
from openai import OpenAI

_openai_client = OpenAI()

def transcribe_audio(audio_path: str) -> str:
    """Transcribes audio file to text using OpenAI Whisper."""
    with open(audio_path, "rb") as f:
        result = _openai_client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
        )
    return result.text

print("✅ Whisper transcription ready.")


In [ ]:
# Cell 12 – Excel score logger
import os
from datetime import datetime
from typing import Optional
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

EXCEL_PATH = "/content/technician_quiz_scores.xlsx"
PASS_THRESHOLD = 4  # out of 5

HEADERS = [
    "Technician Name", "Technician ID", "Server Type",
    "Date", "Time",
    "Q1", "Q2", "Q3", "Q4", "Q5",
    "Score (x/5)", "Score (%)", "Overall Result",
]

def _ensure_workbook():
    """Create the Excel file with a formatted header row if it doesn't exist."""
    if os.path.exists(EXCEL_PATH):
        return
        wb = Workbook()
    ws = wb.active
    ws.title = "Quiz Scores"

    # Header styling
    header_fill   = PatternFill("solid", fgColor="1F4E79")
    header_font   = Font(bold=True, color="FFFFFF", name="Arial", size=11)
    center_align  = Alignment(horizontal="center", vertical="center")
    thin_border   = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"),  bottom=Side(style="thin"),
    )

    for col_idx, header in enumerate(HEADERS, start=1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font      = header_font
        cell.fill      = header_fill
        cell.alignment = center_align
        cell.border    = thin_border

    # Column widths
    col_widths = [22, 16, 14, 12, 10, 8, 8, 8, 8, 8, 14, 12, 16]
    for i, w in enumerate(col_widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w

    ws.row_dimensions[1].height = 20
    ws.freeze_panes = "A2"
    wb.save(EXCEL_PATH)


def save_score_to_excel(
    tech_name: str,
    tech_id: str,
    server_type: str,
    quiz: dict,
    selections: list,
) -> str:
    """
    Appends one row to the Excel score log.
    Returns a status message.
    """
    try:
        _ensure_workbook()
        wb = load_workbook(EXCEL_PATH)
        ws = wb["Quiz Scores"]

        now     = datetime.now()
        correct = 0
        q_results = []

        for i, q in enumerate(quiz["questions"]):
            sel     = selections[i]
            sel_idx = q["options"].index(sel) if sel in q["options"] else None
            passed  = sel_idx == q["correct_index"]
            if passed:
                correct += 1
            q_results.append("Pass" if passed else "Fail")

        score_pct    = int((correct / len(quiz["questions"])) * 100)
        overall      = "PASS" if correct >= PASS_THRESHOLD else "FAIL"

        row = [
            tech_name.strip(),
            tech_id.strip(),
            f"Type {server_type}",
            now.strftime("%Y-%m-%d"),
            now.strftime("%H:%M:%S"),
            *q_results,           # Q1–Q5 Pass/Fail
            f"{correct}/5",
            f"{score_pct}%",
            overall,
        ]

        # Style helpers
        center      = Alignment(horizontal="center")
        thin_border = Border(
            left=Side(style="thin"), right=Side(style="thin"),
            top=Side(style="thin"),  bottom=Side(style="thin"),
        )
        pass_fill   = PatternFill("solid", fgColor="C6EFCE")  # green
        fail_fill   = PatternFill("solid", fgColor="FFC7CE")  # red
        pass_font   = Font(color="276221", bold=True, name="Arial")
        fail_font   = Font(color="9C0006", bold=True, name="Arial")
        normal_font = Font(name="Arial", size=10)

        new_row = ws.max_row + 1
        # Alternate row background
        row_fill = PatternFill("solid", fgColor="EBF3FB") if new_row % 2 == 0 else None

        for col_idx, value in enumerate(row, start=1):
            cell           = ws.cell(row=new_row, column=col_idx, value=value)
            cell.border    = thin_border
            cell.alignment = center
            cell.font      = normal_font
            if row_fill:
                cell.fill = row_fill

        # Colour Q1-Q5 cells (columns 6-10)
        for col_idx, result in enumerate(q_results, start=6):
            cell      = ws.cell(row=new_row, column=col_idx)
            cell.fill = pass_fill if result == "Pass" else fail_fill
            cell.font = pass_font if result == "Pass" else fail_font

        # Colour Overall Result cell (column 13)
        result_cell      = ws.cell(row=new_row, column=13)
        result_cell.fill = pass_fill if overall == "PASS" else fail_fill
        result_cell.font = pass_font if overall == "PASS" else fail_font

        wb.save(EXCEL_PATH)
        return f"✅ Score saved to {EXCEL_PATH}"

    except PermissionError:
        return f"⚠️ Could not save: the file is open in Excel. Close it and resubmit."
    except Exception as e:
        return f"⚠️ Could not save score: {e}"

print("✅ Excel score logger ready.")
print(f"   Scores will be saved to: {EXCEL_PATH}")


In [ ]:
# Cell 13 – Gradio UI helpers
import gradio as gr

NUM_Q = 5

def reset_training_state() -> dict:
    return {
        "tech_name":   "",
        "tech_id":     "",
        "server_type": None,
        "phase":       "idle",
        "quiz":        None,
    }

def _format_quiz_results(quiz: dict, selections: list) -> str:
    lines   = []
    correct = 0
    num_q   = len(quiz["questions"])
    for i, q in enumerate(quiz["questions"], start=1):
        sel        = selections[i - 1]
        sel_idx    = q["options"].index(sel) if sel in q["options"] else None
        is_correct = sel_idx == q["correct_index"]
        if is_correct:
            correct += 1
        lines += [
            f"**Q{i}.** {q['question']}",
            f"- Your answer: {sel or 'No answer'} {'✅' if is_correct else '❌'}",
            f"- Correct: {q['options'][q['correct_index']]}",
            f"- Explanation: {q.get('explanation', 'N/A')}",
            "",
        ]
    pct    = int((correct / num_q) * 100)
    emoji  = "🏆" if pct == 100 else "✅" if pct >= 80 else "📖"
    header = f"## {emoji} Score: **{correct}/{num_q}** ({pct}%)"
    return header + "\n\n" + "\n".join(lines)

def _hidden(**kw): return gr.update(visible=False, **kw)
def _show(**kw):   return gr.update(visible=True,  **kw)

print("✅ UI helpers ready.")


In [ ]:
# Cell 14 – Gradio UI: Training tab callbacks

# Output order (12 items):
# [t_state, training_out, q1, q2, q3, q4, q5, submit_quiz_btn, quiz_results, start_quiz_btn, excel_status]

def _blank(t_state, msg):
    return (t_state, msg,
            _hidden(), _hidden(), _hidden(), _hidden(), _hidden(),
            _hidden(), _hidden(), _hidden(), _hidden())


def start_training_ui(tech_name, tech_id, server_type, t_state):
    """Validate inputs, generate summary, show quiz button."""
    if not (tech_name or "").strip():
        return _blank(t_state, "⚠️ Please enter your name.")
    if not (tech_id or "").strip():
        return _blank(t_state, "⚠️ Please enter your technician ID.")
    if not server_type:
        return _blank(t_state, "⚠️ Please select a server type.")

    try:
        summary = generate_training_summary(server_type)
    except Exception as e:
        return _blank(t_state, f"❌ Error generating summary: {e}")

    t_state = {
        "tech_name":   tech_name.strip(),
        "tech_id":     tech_id.strip(),
        "server_type": server_type,
        "phase":       "summary",
        "quiz":        None,
    }

    msg = (
        f"# 📋 Server Type {server_type} – Assembly Quick Guide\n\n"
        f"{summary}\n\n"
        f"---\n"
        f"*Technician: **{tech_name.strip()}** (ID: {tech_id.strip()})*"
    )

    return (t_state, msg,
            _hidden(), _hidden(), _hidden(), _hidden(), _hidden(),
            _hidden(), _hidden(),
            _show(),   # start_quiz_btn
            _hidden())


def start_quiz_ui(t_state):
    """Build and display 5 MCQs."""
    if not t_state or not t_state.get("server_type"):
        return _blank(t_state, "⚠️ Load training summary first.")

    try:
        quiz = build_quiz(server_type=t_state["server_type"], num_q=NUM_Q, k=14)
    except Exception as e:
        return _blank(t_state, f"❌ Quiz generation failed: {e}\n\nClick **Start Quiz** to retry.")

    t_state["quiz"]  = quiz
    t_state["phase"] = "quiz"
    qs = quiz["questions"]

    intro = (
        f"## 📝 Quiz – Server Type {t_state['server_type']}\n\n"
        f"Answer all {NUM_Q} questions then click **Submit Quiz**."
    )

    return (
        t_state, intro,
        _show(label=f"Q1. {qs[0]['question']}", choices=qs[0]["options"], value=None),
        _show(label=f"Q2. {qs[1]['question']}", choices=qs[1]["options"], value=None),
        _show(label=f"Q3. {qs[2]['question']}", choices=qs[2]["options"], value=None),
        _show(label=f"Q4. {qs[3]['question']}", choices=qs[3]["options"], value=None),
        _show(label=f"Q5. {qs[4]['question']}", choices=qs[4]["options"], value=None),
        _show(),    # submit_quiz_btn
        _hidden(),  # quiz_results
        _hidden(),  # start_quiz_btn
        _hidden(),  # excel_status
    )


def submit_quiz_ui(t_state, a1, a2, a3, a4, a5):
    """Grade quiz, show results, and save to Excel."""
    if not t_state or not t_state.get("quiz"):
        return t_state, _show(value="⚠️ No quiz loaded."), _hidden()

    selections = [a1, a2, a3, a4, a5]
    result_md  = _format_quiz_results(t_state["quiz"], selections)

    # Save to Excel
    excel_msg = save_score_to_excel(
        tech_name   = t_state.get("tech_name", "Unknown"),
        tech_id     = t_state.get("tech_id",   "Unknown"),
        server_type = t_state.get("server_type", "?"),
        quiz        = t_state["quiz"],
        selections  = selections,
    )

    return t_state, _show(value=result_md), _show(value=excel_msg)


def reset_training_ui():
    return (
        reset_training_state(),
        "Enter your name & ID, select a server type, then click **▶ Start Training**.",
        _hidden(choices=[], value=None),
        _hidden(choices=[], value=None),
        _hidden(choices=[], value=None),
        _hidden(choices=[], value=None),
        _hidden(choices=[], value=None),
        _hidden(),
        _hidden(value=""),
        _hidden(),
        _hidden(value=""),
    )


# ── Q&A callbacks ──────────────────────────────────────────────────────────
def qa_text_ui(question, server_type):
    question = (question or "").strip()
    if not question:
        return "", "⚠️ Please type a question."
    st = server_type if server_type in ["A","B","C","D"] else None
    try:
        ans = answer_qa(question, server_type=st)
    except Exception as e:
        ans = f"❌ Error: {e}"
    return "", str(ans)

def qa_voice_ui(audio_path, server_type):
    if audio_path is None:
        return "", "⚠️ No audio received."
    try:
        transcript = transcribe_audio(audio_path)
    except Exception as e:
        return "", f"❌ Transcription error: {e}"
    st = server_type if server_type in ["A","B","C","D"] else None
    try:
        ans = answer_qa(transcript, server_type=st)
    except Exception as e:
        ans = f"❌ Answer error: {e}"
    return transcript, str(ans)

print("✅ All UI callbacks ready.")


In [ ]:
# Cell 15 – Gradio UI: Full app
import gradio as gr

with gr.Blocks(title="AI Technician Trainer", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        "# 🧰 AI Technician Trainer\n"
        "Read a quick assembly guide, then test yourself with a 5-question quiz.\n\n"
        "> Servers supported: **Type A · Type B · Type C · Type D**"
    )

    with gr.Tabs():

        # ── Training Tab ────────────────────────────────────────────────
        with gr.Tab("🎓 Training"):
            t_state = gr.State(reset_training_state())

            # Technician details
            gr.Markdown("### 👤 Technician Details")
            with gr.Row():
                tech_name_input = gr.Textbox(
                    label="Full Name",
                    placeholder="e.g. John Smith",
                    scale=2,
                )
                tech_id_input = gr.Textbox(
                    label="Technician ID",
                    placeholder="e.g. T-1042",
                    scale=1,
                )

            # Server selection
            gr.Markdown("### 🖥️ Server Selection")
            with gr.Row():
                training_server = gr.Radio(
                    choices=["A", "B", "C", "D"],
                    label="Select Server Type",
                    value=None,
                    scale=3,
                )
                with gr.Column(scale=1):
                    btn_start = gr.Button("▶ Start Training", variant="primary")
                    btn_reset = gr.Button("🔄 Reset",         variant="secondary")

            training_out = gr.Markdown(
                "Enter your details, select a server type, then click **▶ Start Training**."
            )

            # Quiz section
            start_quiz_btn  = gr.Button("🏆 Start Quiz (5 MCQs)", variant="primary", visible=False)

            q1 = gr.Radio(label="Q1", choices=[], visible=False)
            q2 = gr.Radio(label="Q2", choices=[], visible=False)
            q3 = gr.Radio(label="Q3", choices=[], visible=False)
            q4 = gr.Radio(label="Q4", choices=[], visible=False)
            q5 = gr.Radio(label="Q5", choices=[], visible=False)

            submit_quiz_btn = gr.Button("📤 Submit Quiz", variant="primary", visible=False)
            quiz_results    = gr.Markdown("", visible=False)
            excel_status    = gr.Markdown("", visible=False)

            _train_outputs = [
                t_state, training_out,
                q1, q2, q3, q4, q5,
                submit_quiz_btn, quiz_results, start_quiz_btn, excel_status,
            ]

            btn_start.click(
                fn=start_training_ui,
                inputs=[tech_name_input, tech_id_input, training_server, t_state],
                outputs=_train_outputs,
            )
            btn_reset.click(
                fn=reset_training_ui,
                outputs=_train_outputs,
            )
            start_quiz_btn.click(
                fn=start_quiz_ui,
                inputs=[t_state],
                outputs=_train_outputs,
            )
            submit_quiz_btn.click(
                fn=submit_quiz_ui,
                inputs=[t_state, q1, q2, q3, q4, q5],
                outputs=[t_state, quiz_results, excel_status],
            )

        # ── Q&A Tab ──────────────────────────────────────────────────────
        with gr.Tab("💬 Q&A"):
            qa_server = gr.Dropdown(
                choices=["(Any)", "A", "B", "C", "D"],
                value="(Any)",
                label="Filter by Server Type (optional)",
            )
            gr.Markdown("### ✏️ Text Q&A")
            qa_in     = gr.Textbox(label="Your question", placeholder="Ask anything about assembly…", lines=2)
            qa_btn    = gr.Button("Ask", variant="primary")
            qa_answer = gr.Markdown()
            qa_btn.click(qa_text_ui, inputs=[qa_in, qa_server], outputs=[qa_in, qa_answer])
            qa_in.submit(qa_text_ui, inputs=[qa_in, qa_server], outputs=[qa_in, qa_answer])

            gr.Markdown("---\n### 🎤 Voice Q&A")
            audio_in       = gr.Audio(sources=["microphone"], type="filepath", label="Record your question")
            voice_btn      = gr.Button("Transcribe & Ask", variant="primary")
            transcript_out = gr.Textbox(label="Transcript", interactive=False)
            voice_answer   = gr.Markdown()
            voice_btn.click(qa_voice_ui, inputs=[audio_in, qa_server], outputs=[transcript_out, voice_answer])

demo.launch(debug=True, share=True)
